# 🧩 Unsupervised Learning — K-Means Clustering & PCA

**A beginner-friendly Colab notebook. No heavy maths — only simple ideas explained in plain words.**

In this notebook you will:
1. Understand what *unsupervised learning* is
2. Learn the **3 tiny maths ideas** behind K-Means (average, distance, squares) — with real numbers, no scary formulas
3. Build K-Means **by hand (from scratch)** and watch it learn step by step 👀
4. Use the real `scikit-learn` K-Means on student data
5. Choose the right number of groups (Elbow method + Silhouette score)
6. Use **PCA** to draw 6-column data on a simple 2D graph

> 💡 **How to use:** Run each cell one by one with **Shift + Enter**. Read the text above a cell first, then run it, then read the output.
> Don't worry if maths is not your strong area — every maths idea here is explained with small numbers you can check on paper.

## 🧠 Part 0 — What is Unsupervised Learning?

In **supervised learning** we gave the computer data **with answers** (e.g. flower species, house price).

In **unsupervised learning** there are **no answers**. We give only the data, and the computer must **find patterns by itself**.

**Real-life picture 🍎🍌🍇:** Someone gives you a basket of mixed fruits with **no labels**. You will still put similar fruits together — round red ones here, long yellow ones there. Nobody told you the names, you just saw *similarity*. That is **clustering**.

| | Supervised | Unsupervised |
|---|---|---|
| Data has answers (labels)? | ✅ Yes | ❌ No |
| Goal | Predict the answer | Find hidden groups / patterns |
| Example | Will the student pass? | Which types of students do we have? |
| Algorithms | KNN, Decision Tree, Regression | **K-Means**, PCA |

### Where is clustering used in real life?
- 🛍️ Grouping customers (budget buyers, premium buyers…) for offers
- 🎓 Grouping students by study habits to give the right support
- 📰 Grouping similar news articles
- 🖼️ Reducing colours in an image (image compression)
- 🚨 Finding unusual behaviour (fraud detection)

### The K-Means idea in one line
> **"Split the data into K groups, so that members of a group are close to each other."**
> Here **K** is the number of groups *we* choose (K = 3 means 3 groups).

## 📦 Step 1 — Import the libraries

- **numpy** → maths on numbers
- **pandas** → tables
- **matplotlib / seaborn** → graphs
- **scikit-learn** → `KMeans`, `PCA`, `StandardScaler`, `silhouette_score`

In [ ]:
import numpy as np                      # maths on numbers and arrays
import pandas as pd                     # tables (DataFrames)
import matplotlib.pyplot as plt         # graphs
import seaborn as sns                   # prettier graphs

from sklearn.cluster import KMeans                  # the K-Means algorithm
from sklearn.preprocessing import StandardScaler    # puts all columns on the same scale
from sklearn.decomposition import PCA               # squeezes many columns into 2 (for drawing)
from sklearn.metrics import silhouette_score        # measures how good the groups are

sns.set_theme(style="whitegrid")        # clean graph style
print("✅ Libraries loaded!")

# 🔢 PART 1 — The 3 tiny maths ideas behind K-Means

You need only **three** ideas. Each one is a school-level idea. Let's do each with real numbers.

---
## 1️⃣ Average (Mean) → finds the "centre" of a group

The **average** is the *balancing point* of some numbers. Add them up, divide by how many.

For points, we do it **separately for x and for y**. The result is called the **centroid** (centre point) of the group.

Example: three points `(1, 2)`, `(2, 4)`, `(3, 6)`
- average of x = (1 + 2 + 3) ÷ 3 = **2**
- average of y = (2 + 4 + 6) ÷ 3 = **4**
- centroid = **(2, 4)**

In [ ]:
group = np.array([[1, 2],
                  [2, 4],
                  [3, 6]])          # 3 points, each has (x, y)

# axis=0 means: calculate the average DOWN each column (x column, y column)
centroid = group.mean(axis=0)

print("Centroid (centre) of the group:", centroid)    # should be [2. 4.]

## 2️⃣ Distance → "how far is this point from that point?"

K-Means keeps asking: **"Which centre is this point closest to?"** So we need a way to measure distance.

Think of a map: your friend's house is **3 steps east** and **4 steps north** of yours. If you could fly straight, how far is it?

This is the famous **3-4-5 triangle** (Pythagoras):

> distance = √( 3² + 4² ) = √( 9 + 16 ) = √25 = **5**

The steps to remember (in plain words):
1. Find the **difference** in each direction (3 and 4)
2. **Square** each one (9 and 16)
3. **Add** them (25)
4. Take the **square root** (5)

With 6 columns (study hours, attendance, …) we do exactly the same: just add 6 squared differences instead of 2. The idea never changes! 🙂

In [ ]:
house_A = np.array([1, 2])      # my house
house_B = np.array([4, 6])      # friend's house

difference = house_B - house_A                   # [3, 4]  -> 3 steps east, 4 steps north
print("Difference in x and y:", difference)

# Step by step, exactly like above
distance_manual = np.sqrt(3**2 + 4**2)           # square both, add, square root
print("Distance (by hand)  :", distance_manual)

# numpy has a ready-made shortcut that does the same thing
distance_numpy = np.linalg.norm(house_B - house_A)
print("Distance (numpy)    :", distance_numpy)

## 3️⃣ Squares → why do we square the gaps?

Later we measure *"how tightly packed are the groups?"* by adding up gaps. Two problems if we add gaps directly:
- A gap of **−2** (left) and **+2** (right) would add to **0** → looks like "no gap" ❌
- We want **big gaps to be punished more** than small gaps

**Squaring fixes both:** squares are never negative, and big numbers become much bigger.

| gap | gap² |
|---|---|
| −2 | 4 |
| +2 | 4 |
| 10 | 100 |

In [ ]:
gaps = np.array([-2, 2])
print("Adding raw gaps   :", gaps.sum(), "  <- 0, misleading!")
print("Adding squared gaps:", (gaps ** 2).sum(), "  <- 8, now the gaps are counted")

# 🛠️ PART 2 — K-Means from scratch (watch the machine learn!)

Now we use those 3 ideas. **K-Means is only 4 steps:**

| Step | What happens | Maths idea used |
|---|---|---|
| **1. Pick** | Choose K starting centres (random guesses) | — |
| **2. Assign** | Every point joins the **nearest** centre | Distance |
| **3. Move** | Each centre moves to the **average** of its group | Average |
| **4. Repeat** | Do steps 2–3 again until centres stop moving | — |

That's the whole algorithm. 🎉 *"Learning" here = centres slowly moving to the right places.*

### Our tiny practice data
15 points that form **3 obvious blobs**: bottom-left, top-right and bottom-right.

In [ ]:
points = np.array([
    [1.0, 1.5], [1.5, 2.0], [2.0, 1.0], [1.8, 1.8], [1.2, 2.2],    # blob 1 (bottom-left)
    [8.0, 8.5], [8.5, 9.0], [9.0, 8.0], [8.2, 8.8], [9.2, 9.2],    # blob 2 (top-right)
    [8.0, 1.5], [8.5, 2.0], [9.0, 1.0], [8.8, 1.8], [7.8, 2.2],    # blob 3 (bottom-right)
])

plt.figure(figsize=(5, 4))
plt.scatter(points[:, 0], points[:, 1], color="grey", s=70)     # [:, 0] = all x values, [:, 1] = all y values
plt.title("Our 15 points (no colours, no labels)")
plt.xlabel("x"); plt.ylabel("y")
plt.show()

### Write the two main steps as small functions

- `assign_to_nearest` → **Step 2**: for each point, measure the distance to every centre and pick the smallest.
- `move_centers` → **Step 3**: for each group, calculate the average (centroid) of its members.

In [ ]:
def assign_to_nearest(points, centers):
    # STEP 2: give every point the number of its nearest centre (0, 1, 2...)
    labels = []
    for p in points:                                              # take one point at a time
        distances = [np.sqrt(np.sum((p - c) ** 2)) for c in centers]   # distance to EACH centre
        labels.append(int(np.argmin(distances)))                  # argmin = position of the smallest distance
    return np.array(labels)


def move_centers(points, labels, centers):
    # STEP 3: move each centre to the average of the points that joined it
    new_centers = []
    for j in range(len(centers)):
        members = points[labels == j]                             # points that belong to centre j
        if len(members) == 0:                                     # nobody joined? then stay where you are
            new_centers.append(centers[j])
        else:
            new_centers.append(members.mean(axis=0))              # the average = new centre
    return np.array(new_centers)

print("Functions ready ✅")

### Run the full loop (Step 4: repeat until nothing changes)

We start with **3 guessed centres** (not perfect on purpose) and record every round so we can draw it.

In [ ]:
def kmeans_from_scratch(points, start_centers, max_rounds=10):
    centers = start_centers.copy()
    history = []                                       # we store each round to draw later
    for round_no in range(max_rounds):
        labels = assign_to_nearest(points, centers)    # STEP 2
        history.append((labels, centers.copy()))       # remember this round
        new_centers = move_centers(points, labels, centers)   # STEP 3
        if np.allclose(new_centers, centers):          # centres stopped moving? we are done
            break
        centers = new_centers                          # STEP 4: repeat with the new centres
    return history

colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

def draw_history(points, history):
    fig, axes = plt.subplots(1, len(history), figsize=(4 * len(history), 3.8))
    axes = np.atleast_1d(axes)
    for i, (ax, (labels, centers)) in enumerate(zip(axes, history), start=1):
        for j in range(len(centers)):
            ax.scatter(points[labels == j, 0], points[labels == j, 1], color=colors[j], s=60)
            ax.scatter(centers[j, 0], centers[j, 1], color=colors[j], marker="X",
                       s=260, edgecolor="black", linewidth=1.5)     # big X = centre
        ax.set_title(f"Round {i}")
    plt.tight_layout()
    plt.show()

# Our 3 starting guesses (the X marks). They are not in the perfect places.
start_centers = np.array([[3.0, 3.0], [7.0, 5.0], [9.0, 5.0]])

history = kmeans_from_scratch(points, start_centers)
print(f"K-Means finished after {len(history)} rounds")
draw_history(points, history)

### 👀 How to read the pictures
- **Big ✖ marks** = the centres. **Dots** = points, coloured by their nearest centre.
- Round 1: the centres are in rough positions, so the groups are **wrong** (look at the right-hand side: the top-right and bottom-right blobs are shared badly).
- Next rounds: each ✖ **moves to the middle of its dots** → dots re-join the nearest ✖ → the groups get better and better.
- Last round: the centres stop moving. **The groups are found!** 🎯

Let's print the final centres. They should be near the middle of each blob.

In [ ]:
final_labels, final_centers = history[-1]
print("Final centres (x, y):")
print(final_centers.round(2))

## ⚡ The same thing with scikit-learn (1 line!)

In real projects we don't write the loops ourselves. `KMeans` from scikit-learn does the same job, faster and smarter.

- `n_clusters=3` → K, the number of groups
- `n_init=10` → run the whole thing **10 times** from different random starts and keep the best result
- `random_state=42` → makes the random choices repeatable
- `.fit(...)` → learn the centres  |  `.labels_` → group number of each point  |  `.cluster_centers_` → the centres

In [ ]:
km_tiny = KMeans(n_clusters=3, n_init=10, random_state=42)
km_tiny.fit(points)

print("Group number of each point:", km_tiny.labels_)
print("\nCentres found by scikit-learn:")
print(km_tiny.cluster_centers_.round(2))
print("\n(The group numbers 0/1/2 may be in a different order than our version — the numbers are just names.)")

# 🎓 PART 3 — Real use case: finding types of students

**Problem:** A college has 400 students and wants to **group them by study habits** to give the right support to each group. Nobody has labelled the students — we want the computer to **discover** the groups.

### 🏗️ Step 2 — Create the student data

To practise, we create 400 students from **4 hidden student types**. The computer will **not be told** about these types. Later we peek at them just to check whether K-Means found them. 🕵️

> In real life you would load your own file: `students = pd.read_csv("students.csv")` (for example data collected from a Google Form).

In [ ]:
rng = np.random.default_rng(42)     # random number generator (seed 42 -> same data every time)

feature_names = ["study_hours_per_day", "attendance_percent", "previous_marks",
                 "sleep_hours", "assignments_done", "screen_hours_per_day"]

# For each hidden type: (typical value of each column, number of students)
student_types = {
    "Toppers":           ([5.0, 93, 85, 7.0, 9.5, 2.0],  90),
    "Steady average":    ([3.0, 80, 62, 7.2, 7.5, 4.0], 130),
    "Distracted":        ([1.2, 62, 45, 6.0, 3.5, 7.5], 100),
    "Talented but lazy": ([1.0, 70, 80, 8.0, 5.0, 6.0],  80),
}
spread = np.array([0.6, 5, 6, 0.6, 1.0, 0.8])    # how much students of one type differ from each other

parts = []
for type_name, (typical, count) in student_types.items():
    data = rng.normal(loc=typical, scale=spread, size=(count, len(feature_names)))   # random values around the typical ones
    part = pd.DataFrame(data, columns=feature_names)
    part["hidden_type"] = type_name              # the secret label (we will NOT use it for clustering)
    parts.append(part)

students = pd.concat(parts, ignore_index=True)                       # join the 4 groups into one table
students = students.sample(frac=1, random_state=42).reset_index(drop=True)   # shuffle the rows

# Keep values in a realistic range and round them
students["study_hours_per_day"]   = students["study_hours_per_day"].clip(0, 8).round(1)
students["attendance_percent"]    = students["attendance_percent"].clip(30, 100).round(0)
students["previous_marks"]        = students["previous_marks"].clip(15, 100).round(0)
students["sleep_hours"]           = students["sleep_hours"].clip(4, 10).round(1)
students["assignments_done"]      = students["assignments_done"].clip(0, 10).round(0)
students["screen_hours_per_day"]  = students["screen_hours_per_day"].clip(0.5, 10).round(1)

students.insert(0, "student_id", range(1, len(students) + 1))
students.head()

## 🔍 Step 3 — Explore the data

Remember: for clustering we use **only the 6 feature columns**. We leave out `student_id` (just a number) and `hidden_type` (that would be cheating!).

In [ ]:
X = students[feature_names]          # ONLY the 6 measurement columns - no answers, no id

print("Shape (rows, columns):", X.shape)
print("Missing values:", X.isnull().sum().sum())
X.describe().round(2)                # count, mean, min, max of every column

### 👁️ Can YOU see groups?

Here are just two of the six columns. Your eyes may notice some blobs. K-Means will do this for **all 6 columns at once**, which no human eye can do. 💪

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.scatter(students["study_hours_per_day"], students["previous_marks"], color="grey", alpha=0.6)
plt.xlabel("Study hours per day")
plt.ylabel("Previous marks")
plt.title("Students (no colours — the computer doesn't know the types)")
plt.show()

## ⚖️ Step 4 — Scaling (very important for K-Means!)

K-Means uses **distance**. Look at our columns:
- `attendance_percent` ranges roughly **40 to 100** (big numbers)
- `sleep_hours` ranges roughly **4 to 9** (small numbers)

If we don't fix this, attendance would **dominate** the distance and sleep would be almost ignored — like measuring one thing in **kilometres** and another in **millimetres**. ❌

**StandardScaler** puts every column on the same footing: after scaling, every column has **average = 0** and **spread (standard deviation) = 1**. A value now simply means *"how far above or below the average"*.

- `+1` → above average, `-1` → below average, `0` → exactly average

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)       # learn the average & spread of each column, then convert

# Compare before and after
compare = pd.DataFrame({
    "mean BEFORE": X.mean().round(2),
    "std BEFORE":  X.std().round(2),
    "mean AFTER":  pd.DataFrame(X_scaled, columns=feature_names).mean().round(2),
    "std AFTER":   pd.DataFrame(X_scaled, columns=feature_names).std().round(2),
})
compare

## 🎯 Step 5 — How many groups? (Choosing K)

K-Means needs **us** to say K. How do we choose? Two helpers:

### Helper 1: Elbow method — using **inertia**
**Inertia** (in plain words): for every student, measure the distance to *their own group's centre*, **square it** (idea #3), and **add everything up**.
- **Small inertia** = students are huddled tightly around their centres = good groups
- **Large inertia** = students are spread far from their centres

⚠️ Inertia **always goes down** when K increases (more groups = smaller distances). With K = 400 (every student alone) it would be 0 — useless!
So we look for the **elbow**: the point where the line **stops dropping fast** and flattens. Like an arm: the bend at the elbow is the sweet spot.

### Helper 2: Silhouette score
For each student, we ask: *"Am I much closer to my own group than to the next nearest group?"*
- Near **+1** → clearly in the right group ✅
- Near **0** → on the border between two groups 😐
- **Negative** → probably in the wrong group ❌

We take the **average over all students**. **Higher is better.**

In [ ]:
k_values = range(2, 10)         # try K = 2, 3, ... 9  (silhouette needs at least 2 groups)
inertias = []
silhouettes = []

for k in k_values:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(X_scaled)
    inertias.append(model.inertia_)                              # total squared distance to own centres
    silhouettes.append(silhouette_score(X_scaled, model.labels_))  # average silhouette

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(k_values, inertias, marker="o")
axes[0].set_title("Elbow method (look for the bend)")
axes[0].set_xlabel("K (number of groups)")
axes[0].set_ylabel("Inertia (smaller = tighter groups)")

axes[1].plot(k_values, silhouettes, marker="s", color="green")
axes[1].set_title("Silhouette score (look for the highest)")
axes[1].set_xlabel("K (number of groups)")
axes[1].set_ylabel("Average silhouette")

plt.tight_layout()
plt.show()

best_k = list(k_values)[int(np.argmax(silhouettes))]
print("K with the highest silhouette score:", best_k)

### 👀 Reading the two graphs
- **Elbow graph:** the line drops quickly up to K = 4, then the drop becomes small → the **elbow is at K = 4**.
- **Silhouette graph:** the highest point is also at **K = 4**.

Both helpers agree → we choose **K = 4**.

> 💬 On messy real-world data the two graphs may not agree, or the elbow may be unclear. Then use common sense and your business knowledge too (e.g. "the college can only run 3 support programs, so K = 3").

## 🏁 Step 6 — Train the final K-Means model (K = 4)

`fit_predict` does two things in one go: learns the centres **and** returns each student's group number.
We save the group number in a new column called `cluster`.

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
students["cluster"] = kmeans.fit_predict(X_scaled)     # learn + give each student a group number

print("How many students in each cluster?")
print(students["cluster"].value_counts().sort_index())

## 🔎 Step 7 — Understand the groups (the most important step!)

The computer only gives numbers: `0, 1, 2, 3`. **It does not tell us what they mean.** Our job is to *read* each group's average values and give it a meaningful name.

⚠️ The numbers `0, 1, 2, 3` are just **names**. On your computer, "cluster 0" may be a different group than on someone else's. So always read the table — don't trust the number.

### Table 1: Average of each group (in real units)

In [ ]:
profile = students.groupby("cluster")[feature_names].mean().round(1)    # average of each column, per cluster
profile

### Table 2: The same thing as a heat map (in "above / below average" units)

Because the columns have different units, it's easier to compare using the **scaled** values:
- 🟥 **Red** = this group is **above** the overall average in that column
- 🟦 **Blue** = this group is **below** the overall average
- White = about average

In [ ]:
# Average of the SCALED values for each cluster ( + means above average, - means below average )
z_table = pd.DataFrame(X_scaled, columns=feature_names).groupby(students["cluster"]).mean()

plt.figure(figsize=(9, 3.5))
sns.heatmap(z_table.round(2), annot=True, cmap="coolwarm", center=0)
plt.title("Cluster profile (red = above average, blue = below average)")
plt.ylabel("Cluster")
plt.show()

### Let the computer describe each group in plain English

We pick the columns where a group is clearly higher (> +0.5) or lower (< −0.5) than average.

In [ ]:
readable = {
    "study_hours_per_day": "study hours",
    "attendance_percent": "attendance",
    "previous_marks": "previous marks",
    "sleep_hours": "sleep",
    "assignments_done": "assignments done",
    "screen_hours_per_day": "screen time",
}

for c in sorted(z_table.index):
    row = z_table.loc[c]
    higher = [readable[f] for f in row.index if row[f] > 0.5]
    lower  = [readable[f] for f in row.index if row[f] < -0.5]
    size = (students["cluster"] == c).sum()
    print(f"Cluster {c}  ({size} students)")
    print("   Higher than average :", ", ".join(higher) or "-")
    print("   Lower than average  :", ", ".join(lower) or "-")
    print()

### ✍️ Now YOU name the groups!

Read the output above and give each cluster a name. For example:

| If a cluster is… | A good name could be |
|---|---|
| High study hours + high marks + low screen time | **Toppers** 🏆 |
| Close to average in everything | **Steady average students** |
| Low study hours + low attendance + high screen time | **Distracted / need support** ⚠️ |
| Low study hours but high previous marks + high sleep | **Talented but not working hard** 💤 |

This is how companies use clustering: a human gives names and decides actions (e.g. extra coaching for the "Distracted" group).

## 🕵️ Step 8 — Peek at the hidden truth (check our work)

Remember the secret `hidden_type` column? K-Means never saw it. Let's compare it with the clusters found.

Each **row** is a cluster. If a row has **one big number and zeros**, then that cluster matches exactly one real student type — K-Means found the real groups on its own!

> In real life you won't have `hidden_type`. We use it here only to prove that the method works.

In [ ]:
check = pd.crosstab(students["cluster"], students["hidden_type"])    # count students for each (cluster, hidden type) pair
display(check)

# How many students landed in the cluster that matches their true type?
matched = check.max(axis=1).sum()               # in each cluster, take the biggest count and add them
print(f"\n{matched} out of {len(students)} students ({matched / len(students):.1%}) were grouped with their true type.")

# 🎨 PART 4 — PCA: Drawing 6 columns on a 2D graph

We have **6 columns**, but a screen only has **2 directions** (x and y). How do we draw all 6 columns together?

### PCA in plain words
**Think of a photograph.** A real 3D object is turned into a 2D photo. A good photographer chooses the **angle that shows the most detail**. A bad angle (e.g. looking at a pencil from its tip) loses information.

**PCA = finding the best "camera angle" for your data.** It creates **new axes** (called PC1 and PC2) which are mixtures of the original columns, chosen so that **as much information as possible is kept**.

- **PC1** = the best single direction (keeps the most information)
- **PC2** = the next best direction

`explained_variance_ratio_` tells us **what percentage of the information each new axis keeps**. For example, 0.60 means "this axis keeps 60% of the information".

> ⚠️ **Important:** We *clustered using all 6 columns*. PCA is used **only to draw** the result in 2D.

In [ ]:
pca = PCA(n_components=2)                 # we want 2 new axes
X_pca = pca.fit_transform(X_scaled)       # convert 6 columns -> 2 columns (PC1, PC2)

print("Information kept by each axis:", (pca.explained_variance_ratio_ * 100).round(1), "%")
print(f"Total information kept in the 2D picture: {pca.explained_variance_ratio_.sum():.1%}")

### Draw the clusters in 2D

Each dot is a student. The colour is the cluster found by K-Means. The big ✖ marks are the cluster centres.

In [ ]:
centers_pca = pca.transform(kmeans.cluster_centers_)       # convert the centres to the same 2D space

plt.figure(figsize=(7, 5.5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=students["cluster"],
                palette="tab10", s=45, alpha=0.8)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], marker="X", s=300,
            color="black", label="Centres")
plt.xlabel("PC1 (best direction)")
plt.ylabel("PC2 (second best direction)")
plt.title("Student clusters shown in 2D using PCA")
plt.legend(title="Cluster")
plt.show()

### What do PC1 and PC2 actually mean?

Each new axis is a **mix of the original columns**. The table below shows how much each column contributes:
- A **big number** (positive or negative, like 0.5 or −0.5) → that column matters a lot for this axis
- A **number close to 0** → that column hardly matters

Columns with the **same sign** move together; columns with **opposite signs** pull in opposite directions.

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=feature_names, columns=["PC1", "PC2"]).round(2)
loadings

**How to read it:** On PC1, study hours, attendance, marks and assignments have one sign, and screen time has the opposite sign. So **PC1 is roughly an "effort / discipline" axis**: students on one side study and attend a lot, students on the other side spend more time on screens.

# 🔮 PART 5 — Use the model on a new student

A new student joins. Which group does he or she belong to?

⚠️ Remember: new data must be **scaled with the same scaler** we used for training (use `transform`, not `fit_transform`).

`kmeans.transform(...)` gives the **distance to every cluster centre**. The student joins the **nearest** centre — exactly Step 2 of the algorithm.

In [ ]:
# New student: study hours, attendance, previous marks, sleep, assignments, screen hours
new_student = pd.DataFrame([[4.8, 90, 80, 7.0, 9, 2.5]], columns=feature_names)

new_scaled = scaler.transform(new_student)             # same scaling as the training data
group = kmeans.predict(new_scaled)[0]                 # which cluster?
distances = kmeans.transform(new_scaled)[0]           # distance to each centre

print("This student belongs to cluster:", group)
print("\nDistance to each cluster centre:")
for c, d in enumerate(distances):
    print(f"  Cluster {c}: {d:.2f}" + ("   <-- nearest" if c == group else ""))

Look at your profile table: does that cluster match what you expect for a student with high marks, high attendance and low screen time? 🙂

# ⚠️ PART 6 — When can K-Means go wrong?

1. **You must choose K** — a wrong K gives wrong groups (use elbow + silhouette + common sense).
2. **Scaling matters** — without scaling, big-number columns dominate.
3. **Bad starting centres** can give a bad result. Let's see it! 👇
4. It likes **round, similar-sized groups**. It struggles with strange shapes.
5. **Outliers** (very unusual points) can pull the centres away.

### Demo: a bad start

We run our from-scratch K-Means on the 15 practice points, but we place all 3 starting centres in the bottom-left corner.

In [ ]:
bad_start = np.array([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]])     # all three centres start in the same corner!

bad_history = kmeans_from_scratch(points, bad_start)
draw_history(points, bad_history[-1:])      # draw only the final round
print("Look: blob 1 got split into 2 groups, while blob 2 and blob 3 got MERGED into one. K-Means got stuck. 😬")

### How do we avoid this?
`scikit-learn` already protects us in two ways:
1. It chooses **smart starting centres** (a method called *k-means++*, which spreads the starting centres far apart)
2. `n_init=10` runs everything **10 times** and keeps the best result

That's why we always write `n_init=10`.

# 🎓 Summary — What you learned

### K-Means in 4 steps
```
1. Pick K starting centres
2. Assign every point to its NEAREST centre      (uses DISTANCE)
3. Move each centre to the AVERAGE of its group  (uses MEAN)
4. Repeat 2 and 3 until the centres stop moving
```

### The recipe in code
```python
scaler  = StandardScaler();  X_scaled = scaler.fit_transform(X)      # 1. scale
km      = KMeans(n_clusters=4, n_init=10, random_state=42)           # 2. create
labels  = km.fit_predict(X_scaled)                                   # 3. learn + get groups
# 4. profile the groups with groupby().mean(), then give them names
```

### Words to remember (plain meaning)
| Word | Meaning |
|---|---|
| **Cluster** | A group of similar points |
| **Centroid / Centre** | The average point of a cluster |
| **K** | Number of clusters (we choose it) |
| **Distance** | How far two points are (straight line) |
| **Inertia** | Total squared distance of points from their own centre (smaller = tighter groups) |
| **Elbow method** | Pick K where the inertia line bends |
| **Silhouette score** | How clearly points belong to their own group (closer to +1 = better) |
| **Scaling** | Putting all columns on the same footing before using distance |
| **PCA** | Finding the best "camera angle" to draw many columns in 2D |
| **PC1, PC2** | The new axes created by PCA |

### Supervised vs Unsupervised — final comparison
| | Supervised (last notebook) | Unsupervised (this notebook) |
|---|---|---|
| Answers given? | Yes | No |
| Evaluate with | Accuracy, MAE, R² | Silhouette, inertia + human judgement |
| Output | A prediction | Groups / patterns |

---

## 📝 Practice Exercises

1. In Step 6, change K to **3** and then **5**. Look at the profile table. Do the groups still make sense?
2. Remove the **scaling** step (use `X` instead of `X_scaled` in `KMeans`). Does the crosstab in Step 8 get worse?
3. Use only **2 columns** (`study_hours_per_day`, `previous_marks`) and cluster again. Draw the result directly (no PCA needed).
4. Load the Iris dataset (`from sklearn.datasets import load_iris`), **hide the species**, run K-Means with K = 3 and compare with the real species using `pd.crosstab`.
5. Change `random_state` from 42 to 7 and 99. Do the cluster *numbers* change? Do the *groups* change?
6. Collect real data from 20–30 classmates with a Google Form (hours of study, sleep, screen time…) and cluster them!

## 🚀 What next?
- **Other clustering methods:** Hierarchical clustering, DBSCAN
- **Neural Networks** — the next big topic: how a network learns using the same "guess → measure error → adjust" idea you saw in the earlier gradient descent demo
- **Anomaly detection** — finding unusual points

*Great job! 🎉*